In [1]:
!pip install mlflow dagshub statsforecast kaggle -q

In [2]:
import os
import dagshub
import mlflow

mlflow.end_run()

dagshub.init(
    repo_owner='tsarc21',
    repo_name='Walmart-Recruiting---Store-Sales-Forecasting',
    mlflow=True
)

print("DagsHub & MLflow connected ✅")

Accessing as lkhar21

Initialized MLflow to track repo "tsarc21/Walmart-Recruiting---Store-Sales-Forecasting"

Repository tsarc21/Walmart-Recruiting---Store-Sales-Forecasting initialized!

DagsHub & MLflow connected ✅


In [ ]:
import os
import shutil

if os.path.exists('kaggle.json'):
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

!kaggle competitions download -c walmart-recruiting-store-sales-forecasting
!unzip -q walmart-recruiting-store-sales-forecasting.zip
!unzip -q train.csv.zip
!unzip -q features.csv.zip
!unzip -q stores.csv.zip

walmart-recruiting-store-sales-forecasting.zip: Skipping, found more recently modified local copy (use --force to force download)
replace features.csv.zip? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import pandas as pd

train = pd.read_csv("train.csv")
features = pd.read_csv("features.csv")
stores = pd.read_csv("stores.csv")

df = train.merge(features, on=["Store", "Date"], how="left")
df = df.merge(stores, on="Store", how="left")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(["Store", "Dept", "Date"])

df.head()

In [ ]:
df_ts = df[["Store", "Dept", "Date", "Weekly_Sales"]].copy()

df_ts["unique_id"] = df_ts["Store"].astype(str) + "_" + df_ts["Dept"].astype(str)
df_ts = df_ts.rename(columns={
    "Date": "ds",
    "Weekly_Sales": "y"
})

df_ts = df_ts[["unique_id", "ds", "y"]]

df_ts.head()

In [ ]:
unique_ids = df_ts["unique_id"].unique()

train_ids = unique_ids[:50]

train_df = df_ts[df_ts["unique_id"].isin(train_ids)].copy()

val_df = train_df.groupby("unique_id").tail(12)
train_df = train_df.drop(val_df.index)

print(train_df.shape, val_df.shape)

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

models = [
    AutoARIMA(
        seasonal=True,
        season_length=52,   # 🔥 weekly yearly seasonality
        max_p=3,
        max_q=3,
        max_P=2,
        max_Q=2,
        d=1,
        D=1
    )
]

sf = StatsForecast(
    models=models,
    freq="W-FRI",
    n_jobs=-1
)

In [ ]:
min_len = 60  # ცოტა მეტი ვიდრე 52 (safety buffer)

valid_ids = (
    train_df.groupby("unique_id")["y"]
    .count()
    .reset_index()
)

valid_ids = valid_ids[valid_ids["y"] >= min_len]["unique_id"]

train_df_clean = train_df[train_df["unique_id"].isin(valid_ids)]
val_df_clean = val_df[val_df["unique_id"].isin(valid_ids)]

print("Original series:", train_df["unique_id"].nunique())
print("Filtered series:", train_df_clean["unique_id"].nunique())

In [ ]:
forecast = sf.forecast(
    df=train_df_clean,
    h=12
)

forecast.head()

In [ ]:
forecast = sf.forecast(df=train_df_clean, h=12)

In [ ]:
display(forecast)

In [ ]:
print(forecast.shape)
print(forecast.columns)

In [ ]:
forecast = forecast.rename(columns={"AutoARIMA": "y_pred"})

In [ ]:
eval_df = val_df_clean.merge(
    forecast,
    on=["unique_id", "ds"],
    how="inner"
)

eval_df.head()

In [ ]:
forecast.head()

In [ ]:
forecast.head()
forecast.shape
forecast.columns

In [ ]:
forecast = forecast.rename(columns={"AutoARIMA": "y_pred"})

In [ ]:
eval_df = val_df_clean.merge(
    forecast,
    on=["unique_id", "ds"],
    how="inner"
)

print(eval_df.shape)
eval_df.head()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(eval_df["y"], eval_df["y_pred"])
rmse = np.sqrt(mean_squared_error(eval_df["y"], eval_df["y_pred"]))

mape = np.mean(np.abs((eval_df["y"] - eval_df["y_pred"]) / eval_df["y"])) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

In [ ]:
baseline = eval_df.copy()
baseline["baseline_pred"] = eval_df.groupby("unique_id")["y"].shift(1)
baseline = baseline.dropna()

baseline_mae = mean_absolute_error(baseline["y"], baseline["baseline_pred"])

improvement = (baseline_mae - mae) / baseline_mae * 100

print("Baseline MAE:", baseline_mae)
print("SARIMA MAE:", mae)
print("Improvement %:", improvement)

In [ ]:
import matplotlib.pyplot as plt

sid = eval_df["unique_id"].iloc[0]
df_plot = eval_df[eval_df["unique_id"] == sid]

plt.figure(figsize=(12,5))
plt.plot(df_plot["ds"], df_plot["y"], label="Actual")
plt.plot(df_plot["ds"], df_plot["y_pred"], label="SARIMA")

plt.legend()
plt.title(f"SARIMA Forecast - {sid}")
plt.xticks(rotation=45)
plt.show()

In [ ]:
import mlflow

mlflow.set_experiment("SARIMA_Experiment")

with mlflow.start_run(run_name="SARIMA_AutoARIMA"):

    mlflow.log_param("model", "AutoARIMA")
    mlflow.log_param("seasonal", True)
    mlflow.log_param("season_length", 52)

    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAPE", mape)
    mlflow.log_metric("improvement_vs_baseline", improvement)

In [ ]:
import joblib

joblib.dump(sf, "sarima_model.pkl")
mlflow.log_artifact("sarima_model.pkl")

In [ ]:
eval_df.head(50).to_csv("sarima_predictions.csv", index=False)
mlflow.log_artifact("sarima_predictions.csv")

In [ ]:
mae = mean_absolute_error(eval_df["y"], eval_df["y_pred"])
rmse = np.sqrt(mean_squared_error(eval_df["y"], eval_df["y_pred"]))
mape = np.mean(np.abs((eval_df["y"] - eval_df["y_pred"]) / eval_df["y"])) * 100

print(mae, rmse, mape)

In [ ]:
import mlflow

mlflow.set_experiment("SARIMA_Experiment")

with mlflow.start_run(run_name="SARIMA"):

    mlflow.log_param("model", "AutoARIMA")
    mlflow.log_param("seasonal", True)
    mlflow.log_param("season_length", 52)

    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAPE", mape)